Zadanie 7

In [1]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split    
from sklearn.metrics import r2_score    

data = fetch_california_housing()   
X = data.data
y = data.target 

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42)

alphas = np.logspace(-4, 4, 100)    

model = RidgeCV(alphas=alphas, cv=5)
model.fit(X_train, y_train)

print("Wybrane alpha:", model.alpha_)   

y_pred = model.predict(X_test)  
r2 = r2_score(y_test, y_pred)
print("Test R²:", r2)

Wybrane alpha: 8.497534359086455
Test R²: 0.5763427800451837


Zadanie 8

In [5]:
from sklearn.linear_model import LassoCV

model_lasso = LassoCV(alphas=alphas, cv=5)
model_lasso.fit(X_train, y_train)

print("Wybrane alpha dla Lasso:", model_lasso.alpha_) 
coef = model_lasso.coef_
num_zeroed = np.sum(np.abs(coef) < 1e-6)
print("Liczba wyzerowanych cech:", num_zeroed)
active_features = np.where(np.abs(coef) >= 1e-6)[0]
print("Aktywne cechy:", [data.feature_names[i] for i in active_features]) 
y_pred_lasso = model_lasso.predict(X_test)
r2_lasso = r2_score(y_test, y_pred_lasso)
print("Test R² dla Lasso:", r2_lasso)





Wybrane alpha dla Lasso: 0.0004430621457583882
Liczba wyzerowanych cech: 0
Aktywne cechy: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Test R² dla Lasso: 0.5764809608952052


Zadanie 12

In [4]:
from sklearn.datasets import make_regression
from sklearn.linear_model import ElasticNetCV   
import pandas as pd

X_synthetic, y_synthetic = make_regression(n_samples=500, n_features=20, n_informative=10, noise=0.1, random_state=42)  
# RidgeCV
ridge_cv_california = RidgeCV(alphas=alphas, cv=5)
ridge_cv_california.fit(X_train, y_train)
ridge_cv_synthetic = RidgeCV(alphas=alphas, cv=5)
ridge_cv_synthetic.fit(X_synthetic, y_synthetic)    
# LassoCV
lasso_cv_california = LassoCV(alphas=alphas, cv=5)
lasso_cv_california.fit(X_train, y_train)
lasso_cv_synthetic = LassoCV(alphas=alphas, cv=5)
lasso_cv_synthetic.fit(X_synthetic, y_synthetic)
# ElasticNetCV
elastic_cv_california = ElasticNetCV(alphas=alphas, cv=5, l1_ratio=0.5)
elastic_cv_california.fit(X_train, y_train)
elastic_cv_synthetic = ElasticNetCV(alphas=alphas, cv=5, l1_ratio=0.5)
elastic_cv_synthetic.fit(X_synthetic, y_synthetic)

results = []
for name, model_california, model_synthetic in [
    ("Ridge", ridge_cv_california, ridge_cv_synthetic),
    ("Lasso", lasso_cv_california, lasso_cv_synthetic),
    ("ElasticNet", elastic_cv_california, elastic_cv_synthetic)
]:
    y_pred_california = model_california.predict(X_test)
    r2_california = r2_score(y_test, y_pred_california)
    num_zeroed_california = np.sum(np.abs(model_california.coef_) < 1e-6)
    
    y_pred_synthetic = model_synthetic.predict(X_synthetic)
    r2_synthetic = r2_score(y_synthetic, y_pred_synthetic)
    num_zeroed_synthetic = np.sum(np.abs(model_synthetic.coef_) < 1e-6)
    
    results.append({
        "Model": name,
        "California R²": r2_california,
        "California Zeroed": num_zeroed_california,
        "Synthetic R²": r2_synthetic,
        "Synthetic Zeroed": num_zeroed_synthetic
    })

results_df = pd.DataFrame(results)
print(results_df)

        Model  California R²  California Zeroed  Synthetic R²  \
0       Ridge       0.576343                  0      0.999999   
1       Lasso       0.576481                  0      0.999970   
2  ElasticNet       0.576490                  0      0.999999   

   Synthetic Zeroed  
0                 0  
1                10  
2                 0  


Dla California Housing najlpepszy model to ElasticNet (0 cech wyzerowanych), a dla syntentycznego: ElasticNet i Ridge (0 zech wyzerowanych)